# Mask R-CNN with COCO 91 Classes (Google Colab)

- **Model**: `maskrcnn_resnet50_fpn_v2` (Torchvision)
- **Dataset**: COCO 2017 (91 classes including background)
- **Features**: Instance Segmentation + Bounding Box + Class Name + Score

**사용 방법**  
1. Runtime → Change runtime type → GPU 선택  
2. 위에서부터 순서대로 셀 실행

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q matplotlib opencv-python-headless pillow requests

import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.transforms import functional as F
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import requests
from io import BytesIO
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device          : {device}")
print(f"PyTorch         : {torch.__version__}")
print(f"Torchvision     : {torchvision.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

## 2. COCO 91개 클래스 정의 (공식)

In [ ]:
# torchvision Mask R-CNN 공식 COCO 91 클래스 (0 = background)
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
    'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'hat', 'backpack', 'umbrella',
    'shoe', 'eye glasses', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'plate', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
    'couch', 'potted plant', 'bed', 'mirror', 'dining table', 'window', 'desk',
    'toilet', 'door', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'blender', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'hair brush'
]

print(f"총 클래스 수          : {len(COCO_CLASSES)} (background 포함)")
print(f"실제 객체 클래스 수   : {len(COCO_CLASSES) - 1}")
print("\n[전체 클래스 리스트]")
print("-" * 50)
for i, name in enumerate(COCO_CLASSES):
    print(f"{i:2d}: {name}")

## 3. 사전학습된 Mask R-CNN 모델 로드 (COCO)

In [ ]:
weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = maskrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.5)
model.to(device)
model.eval()

model_categories = weights.meta["categories"]
print(f"모델 카테고리 수 : {len(model_categories)}")
print("모델 카테고리 예시:", model_categories[:15])

## 4. 테스트 이미지 로드

In [ ]:
def load_image_from_url(url):
    response = requests.get(url, timeout=15)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    return img

# 다양한 객체가 있는 이미지 (원하는 URL로 변경 가능)
image_urls = {
    "street": "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=1280",
    "dog":    "https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/dog1.jpg",
    "people": "https://images.unsplash.com/photo-1529156069898-49953e39b3ac?w=1280"
}

# 사용할 이미지 선택
image = load_image_from_url(image_urls["street"])
print(f"이미지 크기: {image.size}")

preprocess = weights.transforms()
img_tensor = preprocess(image).to(device)

## 5. Inference 실행

In [ ]:
with torch.no_grad():
    prediction = model([img_tensor])[0]

print(f"탐지된 객체 수: {len(prediction['labels'])}")

## 6. 결과 시각화 + 클래스별 통계

In [ ]:
def visualize_and_analyze(image, prediction, score_threshold=0.5, classes=COCO_CLASSES):
    boxes = prediction['boxes'].cpu()
    labels = prediction['labels'].cpu()
    scores = prediction['scores'].cpu()
    masks = prediction['masks'].cpu()

    # 임계값 필터링
    keep = scores >= score_threshold
    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]
    masks = masks[keep]

    # 원본 이미지를 uint8 tensor로 변환
    img_tensor = F.to_tensor(image) * 255
    img_tensor = img_tensor.to(torch.uint8)

    # 마스크 이진화
    binary_masks = (masks > 0.5).squeeze(1)

    # 색상 팔레트
    colors = plt.cm.tab20(np.linspace(0, 1, 20))[:, :3] * 255
    colors = [tuple(map(int, c)) for c in colors]

    # 라벨 텍스트
    labels_text = [
        f"{classes[label] if label < len(classes) else f'cls_{label}'}: {score:.2f}"
        for label, score in zip(labels, scores)
    ]

    # Bounding Box 그리기
    result = draw_bounding_boxes(
        img_tensor, boxes,
        labels=labels_text,
        colors=[colors[i % len(colors)] for i in range(len(boxes))],
        width=3,
        font_size=16
    )

    # Segmentation Mask 그리기
    result = draw_segmentation_masks(
        result, binary_masks,
        alpha=0.45,
        colors=[colors[i % len(colors)] for i in range(len(binary_masks))]
    )

    # 클래스별 카운트
    class_counts = Counter([
        classes[l] if l < len(classes) else f"cls_{l}"
        for l in labels.numpy()
    ])

    return result.permute(1, 2, 0).numpy(), class_counts, labels_text


result_img, class_counts, labels_text = visualize_and_analyze(
    image, prediction, score_threshold=0.5
)

# 시각화
plt.figure(figsize=(18, 9))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Image", fontsize=14)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(result_img)
plt.title(f"Mask R-CNN (COCO 91 classes) | Detected: {len(labels_text)}", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. 상세 탐지 결과 + 클래스 통계

In [ ]:
print("=" * 70)
print(f"{'Class':<20} {'Score':<10} {'Bounding Box (x1, y1, x2, y2)'}")
print("=" * 70)

scores = prediction['scores'].cpu().numpy()
labels = prediction['labels'].cpu().numpy()
boxes = prediction['boxes'].cpu().numpy()

for score, label, box in zip(scores, labels, boxes):
    if score >= 0.5:
        class_name = COCO_CLASSES[label] if label < len(COCO_CLASSES) else f"class_{label}"
        print(f"{class_name:<20} {score:.3f}      {box.round(1)}")

print("\n" + "=" * 70)
print("클래스별 탐지 통계")
print("=" * 70)
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<20}: {cnt}개")

## 8. 전체 COCO 91개 클래스 확인

In [ ]:
print("[ COCO 91 Classes Full List ]")
print("-" * 50)
for i, name in enumerate(COCO_CLASSES):
    print(f"{i:2d}: {name}")

## (옵션) 자신의 이미지로 테스트하기

Colab 왼쪽 파일 탭에서 이미지를 업로드한 후 아래 코드를 사용하세요.

In [ ]:
# 자신의 이미지로 테스트할 때 사용
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# image = Image.open(image_path).convert("RGB")

# img_tensor = preprocess(image).to(device)
# with torch.no_grad():
#     prediction = model([img_tensor])[0]

# result_img, class_counts, labels_text = visualize_and_analyze(image, prediction)
# plt.figure(figsize=(12, 8))
# plt.imshow(result_img)
# plt.axis('off')
# plt.show()

---

### 사용 팁

| 항목 | 설명 |
|------|------|
| **모델** | `maskrcnn_resnet50_fpn_v2` (COCO 사전학습) |
| **클래스** | 91개 (background + 80 객체) |
| **점수 임계값** | `score_threshold=0.5` (원하면 0.3~0.7로 조절) |
| **GPU 권장** | Runtime → Change runtime type → T4 GPU |

문제가 발생하면 Runtime을 재시작한 후 처음부터 다시 실행해 주세요.